# S03 — Losses and Gradient Descent

**Week 2 · Module 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s03_losses_and_gradient_descent.ipynb)

Every cell below is a worked example from the [S03 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s03/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s03.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s03.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## Regression losses compared: MSE, MAE, Huber


*Expected output starts with:* `true line: w = 2.00, b = 1.00   (8 of 100 labels shifted by +15)`


In [ ]:
import numpy as np

np.random.seed(0)

# True line y = 2x + 1, Gaussian noise -- then 8 of 100 labels corrupted upward
n = 100
x = np.random.uniform(-2, 2, n)
y = 2 * x + 1 + 0.3 * np.random.randn(n)
y[:8] += 15.0

def fit(loss_grad, lr=0.05, steps=4000):
    """Gradient descent on w, b for a loss with per-example residual gradient loss_grad."""
    w, b = 0.0, 0.0
    for _ in range(steps):
        r = w * x + b - y                 # residuals, (100,)
        g = loss_grad(r)                  # dL/dr per example
        w -= lr * np.mean(g * x)
        b -= lr * np.mean(g)
    return w, b

delta = 1.0
losses = {
    "MSE    (grad 2r)":              lambda r: 2 * r,
    "MAE    (grad sign(r))":         lambda r: np.sign(r),
    "Huber  (grad clip(r, +-1))":    lambda r: np.clip(r, -delta, delta),
}
print(f"true line: w = 2.00, b = 1.00   (8 of {n} labels shifted by +15)")
for name, g in losses.items():
    w, b = fit(g)
    print(f"{name:<28} fitted w = {w:.4f}, b = {b:.4f}")

## Why not MSE for classification?


*Expected output starts with:* `z (logit)   p=sigmoid(z)   |dL/dz| MSE   |dL/dz| CE`


In [ ]:
import numpy as np

np.random.seed(0)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# One example with true label y = 1, prediction p = sigmoid(z).
# MSE loss:            L = (p - y)^2        -> dL/dz = 2 * (p - y) * p * (1 - p)
# Cross-entropy loss:  L = -log(p)          -> dL/dz = p - y
print("z (logit)   p=sigmoid(z)   |dL/dz| MSE   |dL/dz| CE")
for z in [-8.0, -4.0, -2.0, 0.0, 2.0]:
    p = sigmoid(z)
    g_mse = abs(2 * (p - 1) * p * (1 - p))
    g_ce = abs(p - 1)
    print(f"{z:>6.1f}     {p:10.4f}     {g_mse:9.6f}    {g_ce:8.4f}")

## The overflow you will hit, and log-sum-exp


*Expected output starts with:* `naive softmax:  [ 0. nan  0.]`


In [ ]:
import numpy as np

np.random.seed(0)

def softmax_naive(z):
    e = np.exp(z)
    return e / e.sum()

def softmax_stable(z):
    e = np.exp(z - z.max())         # shift so the largest logit is 0
    return e / e.sum()

def log_softmax_stable(z):
    m = z.max()
    return z - (m + np.log(np.exp(z - m).sum()))   # log-sum-exp trick

# Logits of this size appear routinely in real training runs
z = np.array([15.0, 1000.0, -8.0])

with np.errstate(over="ignore", invalid="ignore"):   # silence the warnings; inspect the values
    naive = softmax_naive(z)
print(f"naive softmax:  {naive}")
print(f"stable softmax: {softmax_stable(z)}")

# The same failure hits the loss. True class = 0 (logit 15.0):
with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
    loss_naive = -np.log(softmax_naive(z)[0])
loss_stable = -log_softmax_stable(z)[0]
print(f"naive  cross-entropy for class 0: {loss_naive}")
print(f"stable cross-entropy for class 0: {loss_stable:.4f}")

## Gradient descent and the learning rate


*Expected output starts with:* `lr = 0.005   loss at steps 0/25/50/75/100: 62.5, 38.93, 30.29, 23.57, 18.35`


In [ ]:
import numpy as np

np.random.seed(0)

# A 2-D quadratic bowl, deliberately ill-conditioned: steep in w[1], shallow in w[0]
# L(w) = 0.5 * (w[0]^2 + 25 * w[1]^2); grad = [w[0], 25 * w[1]]
def loss(w):
    return 0.5 * (w[0] ** 2 + 25 * w[1] ** 2)

def grad(w):
    return np.array([w[0], 25 * w[1]])

for lr in [0.005, 0.06, 0.081]:
    w = np.array([10.0, 1.0])
    trace = []
    for step in range(101):
        if step % 25 == 0:
            trace.append(f"{loss(w):.4g}")
        w = w - lr * grad(w)
    print(f"lr = {lr:<6}  loss at steps 0/25/50/75/100: {', '.join(trace)}")

## Beyond the bowl: non-convexity


*Expected output starts with:* `start w0 = -1.5 -> converged to w = -1.0356, loss = -0.3054`


In [ ]:
import numpy as np

np.random.seed(0)

# A non-convex loss: two valleys, the left one deeper
# L(w) = (w^2 - 1)^2 + 0.3*w;  dL/dw = 4*w*(w^2 - 1) + 0.3
def L(w):
    return (w ** 2 - 1) ** 2 + 0.3 * w

def dL(w):
    return 4 * w * (w ** 2 - 1) + 0.3

for w0 in [-1.5, 0.5, 1.5]:
    w = w0
    for _ in range(500):
        w -= 0.01 * dL(w)
    print(f"start w0 = {w0:>4.1f} -> converged to w = {w:.4f}, loss = {L(w):.4f}")

## Going deeper


*Expected output starts with:* `logistic regression        test accuracy = 0.8847   mean confidence = 0.8755`


In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

# Ground truth is genuinely noisy: P(y=1 | x) = sigmoid(2x - 1)
def sample(n, rng):
    x = rng.uniform(-3, 3, size=(n, 1)).astype(np.float32)
    p = 1 / (1 + np.exp(-(2 * x - 1)))
    yl = (rng.uniform(size=p.shape) < p).astype(np.float32)
    return torch.from_numpy(x), torch.from_numpy(yl)

X_tr, y_tr = sample(200, np.random.default_rng(0))
X_te, y_te = sample(20000, np.random.default_rng(1))

def train(model, steps, lr=0.05):
    torch.manual_seed(0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        loss = nn.functional.binary_cross_entropy_with_logits(model(X_tr), y_tr)
        loss.backward()
        opt.step()
    return model

small = train(nn.Linear(1, 1), steps=2000)
big = train(nn.Sequential(nn.Linear(1, 64), nn.ReLU(),
                          nn.Linear(64, 64), nn.ReLU(),
                          nn.Linear(64, 1)), steps=6000)

for name, model in [("logistic regression", small), ("64-unit MLP, trained long", big)]:
    with torch.no_grad():
        p = torch.sigmoid(model(X_te)).numpy().ravel()
    yv = y_te.numpy().ravel()
    pred = p > 0.5
    acc = np.mean(pred == (yv > 0.5))
    conf = np.mean(np.where(pred, p, 1 - p))    # confidence in the predicted class
    print(f"{name:<26} test accuracy = {acc:.4f}   mean confidence = {conf:.4f}")

## Try it yourself

1. In the MSE-vs-CE table, add a column for the absolute error loss `L = |p - y|` (gradient through the sigmoid). Does it fix MSE's vanishing-gradient problem at `z = -8`? Why or why not?
2. In the outlier experiment, vary the corruption: 0, 4, 16, and 32 bad labels out of 100. Plot (or tabulate) each loss's fitted intercept against corruption rate. Where does Huber start to break down?
3. Find the largest three-decimal learning rate for which the quadratic-bowl run still converges, and verify it approaches the theoretical bound `2/25`. Then change the 25 to 100 and predict the new bound before running.
4. Write `cross_entropy(logits, y)` for a batch (2-D logits array, integer label vector) using the log-sum-exp trick, and test it on logits containing `1000.0`. Compare against `scipy.special.log_softmax` or PyTorch if you have them available.
5. In the double-well experiment, add momentum: `v = 0.9 * v - 0.01 * dL(w); w += v`. Does the run starting at `w0 = 1.5` now escape into the deeper left valley? Explain what momentum changed.
6. In the calibration experiment, create a separate validation set with a new random seed. Fit `T` on that validation set by minimizing cross-entropy, freeze it, and report calibration metrics on the untouched test set. How large is the fitted `T`, and how much does test calibration improve?
7. Batch-size experiment: on the XOR dataset from [section 1.1]({{ '/readings/ch1/why-deep-learning-now/' | relative_url }}), compare full-batch gradient descent with minibatches of 10 at the same learning rate. Which reaches 100% accuracy in fewer *examples processed*?


---

Full discussion of everything above: [S03 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s03/).
